# 第6回 GARCH

このノートブックは、スライドと同じ順に並んでいます。
コードはすべて完成しているので、上から順に実行して、結果を読んでください。

- 0節：セットアップ（授業の最初に実行する）
- A節：理論をシミュレーションで確かめる
- B節：実データを解析する
- C節：設定を変えて回し直す

今日の問い：値を予測できない系列でも、予測できるものはあるか。

## 0. セットアップ

授業が始まったら、まずこのセルを実行してください。
パッケージのインストールに少し時間がかかります。終わるまで、次のセルは動きません。

In [ ]:
# OSのコードネームを確認（noble = Ubuntu 24.04 など）
system("lsb_release -cs", intern = TRUE)

options(
  repos = c(CRAN = "https://packagemanager.posit.co/cran/__linux__/noble/2026-09-01"),
  HTTPUserAgent = sprintf("R/%s R (%s)", getRversion(),
    paste(getRversion(), R.version["platform"], R.version["arch"], R.version["os"]))
)
install.packages(c("tsibble", "feasts", "rugarch"))
sessionInfo()

## A. 理論をシミュレーションで確かめる

In [ ]:
library(tsibble)
library(feasts)
library(ggtime)   # ACFの図（autoplot）
library(dplyr)
library(ggplot2)
options(repr.plot.width = 8, repr.plot.height = 4.5)
invisible(Sys.setlocale("LC_TIME", "C"))   # 図の月の名前を英語にする

`rugarch`パッケージは、`library()`で読み込まず、`rugarch::関数名`の形で呼びます。
読み込むと、ほかのパッケージの同じ名前の関数（`report`など）が隠れるからです。

### A-1. 無相関でも、独立とは限らない

長さ1500の系列を2本作ります。同じ乱数`z`から作ります。

- `1 iid noise`：標準正規分布の乱数`z`を、そのまま並べたもの。iid（独立同分布）です。
- `2 GARCH(1,1)`：`z`に、日によって変わる倍率$\sigma_t$をかけたもの。作り方はA-4で説明します。

In [ ]:
# GARCH(1,1) の系列を作る
#   w_t = sigma_t * z_t,   sigma_t^2 = omega + alpha * w_{t-1}^2 + beta * sigma_{t-1}^2
# z は iid のノイズ（平均0、分散1）。beta = 0 なら ARCH(1) になる
sim_garch <- function(z, omega, alpha, beta = 0) {
  n <- length(z)
  s2 <- numeric(n)                           # 条件つき分散 sigma_t^2
  w <- numeric(n)
  s2[1] <- omega / (1 - alpha - beta)        # 最初は、長い目で見た分散から始める
  w[1] <- sqrt(s2[1]) * z[1]
  for (t in 2:n) {
    s2[t] <- omega + alpha * w[t - 1]^2 + beta * s2[t - 1]
    w[t] <- sqrt(s2[t]) * z[t]
  }
  data.frame(t = 1:n, w = w, sigma = sqrt(s2))
}

In [ ]:
set.seed(1)
n <- 1500
z <- rnorm(n)
g <- sim_garch(z, omega = 0.05, alpha = 0.15, beta = 0.80)
sim <- tsibble(t = rep(1:n, 2),
               series = rep(c("1 iid noise", "2 GARCH(1,1)"), each = n),
               w = c(z, g$w),
               key = series, index = t)
ggplot(sim, aes(t, w)) +
  geom_line() +
  facet_wrap(~series, ncol = 1) +
  labs(x = "Time t (step)", y = "Value w_t")

2本とも、0のまわりを動きます。下の系列には、大きく揺れる時期と静かな時期があります。

平均、分散、ACFを比べます。

In [ ]:
sim |>
  as_tibble() |>
  group_by(series) |>
  summarise(mean = mean(w), variance = var(w))

In [ ]:
sim |>
  ACF(w, lag_max = 30) |>
  autoplot() +
  labs(x = "Lag h (step)", y = "Sample ACF of w_t")

どちらも、平均はほぼ0、分散はほぼ1で、ACFはほとんどのラグで破線の中にあります。
第1回の定義では、どちらもホワイトノイズです。

次に、2乗$w_t^2$のACFを見ます。2乗は、その時点でどれだけ大きく動いたかを表します。

In [ ]:
sim |>
  ACF(w^2, lag_max = 30) |>
  autoplot() +
  labs(x = "Lag h (step)", y = "Sample ACF of w_t^2")

- iidの系列：2乗のACFも、破線の中にあります。独立なら、2乗どうしも独立だからです。
- GARCHの系列：2乗に、正の自己相関が長く残ります。無相関ですが、独立ではありません。

大きく揺れる時期と静かな時期がかたまって現れることを、ボラティリティの集中（volatility clustering）と呼びます。

2本目を作るときに使った倍率$\sigma_t$を描きます。$\sigma_t$をボラティリティ（volatility）と呼びます。

In [ ]:
ggplot(g, aes(t)) +
  geom_line(aes(y = abs(w)), colour = "grey60") +
  geom_line(aes(y = sigma), colour = "red") +
  labs(x = "Time t (step)", y = "|w_t| (grey) and sigma_t (red)")

灰色が$|w_t|$、赤が$\sigma_t$です。$\sigma_t$が大きい時期に、$|w_t|$も大きくなっています。

### A-2. ARCH(1)

いちばん簡単なモデルは、分散が1時点前の値の2乗で決まるモデルです。これをARCH(1)と呼びます（autoregressive conditional heteroskedasticity）。

$$
w_t = \sigma_t z_t, \qquad \sigma_t^2 = \omega + \alpha\, w_{t-1}^2
$$

$z_t$は、平均0、分散1のiidなノイズです。理論では、次のことが成り立ちます。

- $w_t$はホワイトノイズで、分散は$\omega / (1 - \alpha)$
- $w_t^2$のACFは$\alpha^h$（AR(1)と同じ形）

$\omega = 0.7$、$\alpha = 0.3$で長さ50000の系列を作り、確かめます。

In [ ]:
# ベクトルの標本ACFを描く
plot_acf <- function(x, lag_max = 30, ylab = "Sample ACF") {
  tsibble(t = seq_along(x), x = x, index = t) |>
    ACF(x, lag_max = lag_max) |>
    autoplot() +
    labs(x = "Lag h (step)", y = ylab)
}

In [ ]:
set.seed(1)
a1 <- sim_garch(rnorm(50000), omega = 0.7, alpha = 0.3)
c(variance_sample = var(a1$w), variance_theory = 0.7 / (1 - 0.3))
theory <- data.frame(lag = 1:10, acf = 0.3^(1:10))
plot_acf(a1$w^2, lag_max = 10, ylab = "ACF of w_t^2 (bar: sample, red: theory)") +
  geom_point(data = theory, aes(lag, acf), colour = "red", size = 2)

棒（標本）が、赤い点（理論）の近くにあります。

### A-3. ARCH効果の検定

2乗に自己相関があることを、ARCH効果（ARCH effect）と呼びます。ARCH効果があるかどうかを検定します。

2乗$w_t^2$を、自分の過去$q$個に回帰します。

$$
w_t^2 = a_0 + a_1 w_{t-1}^2 + \dots + a_q w_{t-q}^2 + u_t
$$

帰無仮説は「ARCH効果がない（$a_1 = \dots = a_q = 0$）」です。
帰無仮説のもとで、回帰に使った観測の数と決定係数$R^2$の積は、近似的に自由度$q$のカイ2乗分布に従います。
これをARCH-LM検定（ラグランジュ乗数検定、Lagrange multiplier test）と呼びます。

In [ ]:
# ARCH-LM検定：2乗を、自分の過去 q 個に回帰する
arch_lm <- function(e, q = 5) {
  X <- embed(e^2, q + 1)                    # 1列目が今の値、2列目以降が 1〜q 時点前の値
  r2 <- summary(lm(X[, 1] ~ X[, -1]))$r.squared
  statistic <- nrow(X) * r2
  c(r_squared = r2, statistic = statistic,
    p_value = pchisq(statistic, df = q, lower.tail = FALSE))
}

A-1の2本に使います。

In [ ]:
rbind(iid   = arch_lm(z),
      garch = arch_lm(g$w))

第4回のリュング・ボックス検定を、2乗に使うこともできます。
第4回では`ljung_box`で計算しました。ここでは、ベクトルにそのまま使える`Box.test()`で、同じ検定をします（`fitdf`が、第4回の`dof`にあたります）。

In [ ]:
c(iid   = Box.test(z^2, lag = 10, type = "Ljung-Box")$p.value,
  garch = Box.test(g$w^2, lag = 10, type = "Ljung-Box")$p.value)

1回だけでは、たまたまかもしれません。長さ500の系列を1000本ずつ作り、有意水準5%で棄却された割合を出します。

- `iid_normal`：正規分布のiidなノイズ
- `iid_t`：自由度5のt分布のiidなノイズ（極端な値が出やすい）
- `garch`：GARCH(1,1)

In [ ]:
set.seed(1)
reject_rate <- function(make) {
  mean(replicate(1000, arch_lm(make())[["p_value"]] < 0.05))
}
c(iid_normal = reject_rate(function() rnorm(500)),
  iid_t      = reject_rate(function() rt(500, df = 5)),
  garch      = reject_rate(function() sim_garch(rnorm(500), 0.05, 0.15, 0.80)$w))

- iidなら、棄却される割合はほぼ5%です。極端な値が出やすい分布でも、iidであれば変わりません。
- GARCHの系列では、ほとんどの場合に棄却されます。

### A-4. GARCH(1,1)

GARCH(1,1)は、分散の式に、1時点前の分散$\sigma_{t-1}^2$を加えたモデルです（generalized ARCH）。

$$
w_t = \sigma_t z_t, \qquad \sigma_t^2 = \omega + \alpha\, w_{t-1}^2 + \beta\, \sigma_{t-1}^2
$$

- $\alpha$：昨日の揺れの2乗に、どれだけ反応するか
- $\beta$：昨日の分散を、どれだけ引き継ぐか
- $\alpha + \beta < 1$のとき定常で、長い目で見た分散は$\omega / (1 - \alpha - \beta)$

係数は、最尤法で推定します。`rugarch`パッケージを使います。

In [ ]:
# GARCH(1,1) を最尤法で当てはめる。使うパッケージ（rugarch）は、この関数の中だけで呼ぶ
#   ar   ：平均のモデルのARの次数（0なら定数だけ）
#   dist ：ノイズ z の分布（"norm"：正規分布、"std"：t分布）
fit_garch <- function(x, ar = 0, dist = "norm") {
  spec <- rugarch::ugarchspec(
    variance.model = list(model = "sGARCH", garchOrder = c(1, 1)),
    mean.model = list(armaOrder = c(ar, 0), include.mean = TRUE),
    distribution.model = dist)
  fit <- rugarch::ugarchfit(spec, x, solver = "hybrid")
  est <- rugarch::coef(fit)
  loglik <- rugarch::likelihood(fit)
  list(coef = est,                                    # 推定値（mu, ar1, omega, alpha1, beta1, shape）
       se = fit@fit$matcoef[, 2],                     # 標準誤差
       e = as.numeric(rugarch::residuals(fit)),       # 平均のモデルの残差
       sigma = as.numeric(rugarch::sigma(fit)),       # 推定したボラティリティ sigma_t
       loglik = loglik,                               # 対数尤度
       aic = -2 * loglik + 2 * length(est))           # AIC
}

$\omega = 0.05$、$\alpha = 0.15$、$\beta = 0.80$で長さ3000の系列を作り、係数を推定します。

In [ ]:
set.seed(1)
g <- sim_garch(rnorm(3000), omega = 0.05, alpha = 0.15, beta = 0.80)
fit <- fit_garch(g$w)
round(cbind(true = c(mu = 0, omega = 0.05, alpha1 = 0.15, beta1 = 0.80),
            estimate = fit$coef, se = fit$se), 3)

推定値は、本当の値の近くにあります。

この系列は自分で作ったので、本当の$\sigma_t$が分かっています。推定した$\sigma_t$と比べます。

In [ ]:
ggplot(data.frame(t = g$t, true = g$sigma, estimated = fit$sigma), aes(t)) +
  geom_line(aes(y = true), colour = "grey50", linewidth = 1) +
  geom_line(aes(y = estimated), colour = "red") +
  labs(x = "Time t (step)", y = "sigma_t (grey: true, red: estimated)")
cor(g$sigma, fit$sigma)

$\sigma_t$は、直接は観測できません。観測できるのは$w_t$だけです。それでも、$w_t$の列から$\sigma_t$をほぼ取り出せています。

### A-5. 分散を予測する

GARCHでは、今日までの値から、明日の分散$\sigma_{T+1}^2$が式で決まります。$h$時点先の分散の予測は、次の式になります。

$$
\sigma_{T+h \mid T}^2 = \bar{\sigma}^2 + (\alpha + \beta)^{h-1} \left( \sigma_{T+1}^2 - \bar{\sigma}^2 \right), \qquad \bar{\sigma}^2 = \frac{\omega}{1 - \alpha - \beta}
$$

先になるほど、長い目で見た分散$\bar{\sigma}^2$に近づきます。近づく速さは$\alpha + \beta$で決まります。

In [ ]:
# 平均のモデル（定数、またはAR(1)）の残差
mean_resid <- function(x, est) {
  phi <- if ("ar1" %in% names(est)) est[["ar1"]] else 0
  d <- x - est[["mu"]]
  d - phi * c(0, d[-length(d)])
}
# 残差 e と係数から、条件つき分散 sigma_t^2 を順に計算する（var0 は最初の時点の分散）
garch_filter <- function(e, est, var0) {
  s2 <- numeric(length(e))
  s2[1] <- var0
  for (t in 2:length(e)) {
    s2[t] <- est[["omega"]] + est[["alpha1"]] * e[t - 1]^2 + est[["beta1"]] * s2[t - 1]
  }
  s2
}
# h時点先の分散の予測（s2_next は1時点先の分散）
garch_ahead <- function(s2_next, est, h) {
  persistence <- est[["alpha1"]] + est[["beta1"]]
  var_long <- est[["omega"]] / (1 - persistence)
  var_long + persistence^(h - 1) * (s2_next - var_long)
}

A-4の系列で、最も静かだった時点と、最も荒れていた時点から、100時点先までを予測します。

In [ ]:
est <- fit$coef
s2 <- fit$sigma^2
ahead <- rbind(
  data.frame(h = 1:100, start = "from the calmest time",
             sd = sqrt(garch_ahead(min(s2), est, 1:100))),
  data.frame(h = 1:100, start = "from the most volatile time",
             sd = sqrt(garch_ahead(max(s2), est, 1:100))))
sd_long <- sqrt(est[["omega"]] / (1 - est[["alpha1"]] - est[["beta1"]]))
ggplot(ahead, aes(h, sd, colour = start)) +
  geom_line() +
  geom_hline(yintercept = sd_long, linetype = "dashed") +
  labs(x = "Horizon h (step)", y = "Forecast of sigma", colour = NULL)
c(persistence = est[["alpha1"]] + est[["beta1"]], sd_long = sd_long)

どちらから始めても、破線（長い目で見た標準偏差）に近づきます。分散を予測できるのは、近い先だけです。

### A-6. 当てはまりを確かめる

モデルが合っていれば、残差をボラティリティで割った値$\hat{z}_t = \hat{w}_t / \hat{\sigma}_t$は、iidなノイズに近いはずです。
$\hat{z}_t$を標準化残差（standardized residual）と呼びます。

標準化残差の2乗に、自己相関が残っていないかを確かめます。

In [ ]:
z_hat <- fit$e / fit$sigma
plot_acf(z_hat^2, ylab = "Sample ACF of squared standardized residuals")
rbind(w     = arch_lm(g$w),
      z_hat = arch_lm(z_hat))

もとの系列$w_t$ではARCH効果が検出され、標準化残差では検出されません。揺れの大きさの従属性を、モデルが説明しています。

### A-7. 裾の重い分布

正規分布より極端な値が出やすい分布を、裾の重い分布（heavy-tailed distribution）と呼びます。
裾の重さは、尖度（kurtosis）で測ります。正規分布の尖度は3で、3より大きければ、正規分布より裾が重いと読みます。

In [ ]:
# 尖度（正規分布なら3）
kurtosis <- function(x) mean((x - mean(x))^4) / mean((x - mean(x))^2)^2

A-2とA-4の系列は、ノイズ$z_t$を正規分布から作りました。それでも、$w_t$の尖度は3を超えます。
ARCH(1)の尖度は、理論では$3(1 - \alpha^2)/(1 - 3\alpha^2)$です。

In [ ]:
c(arch1_sample = kurtosis(a1$w), arch1_theory = 3 * (1 - 0.3^2) / (1 - 3 * 0.3^2))
c(garch_w = kurtosis(g$w), garch_z_hat = kurtosis(z_hat))

分散の大きい時期と小さい時期が混ざるので、全体として見ると、極端な値が正規分布より多くなります。
標準化残差$\hat{z}_t$の尖度は、ほぼ3に戻ります。

次に、ノイズ$z_t$そのものの裾が重い場合を作ります。自由度5のt分布を、分散が1になるように縮めて使います。
この系列に、$z_t$を正規分布としたGARCHと、t分布としたGARCHを当てはめます。

In [ ]:
set.seed(1)
z_t <- rt(3000, df = 5) * sqrt(3 / 5)     # 自由度5のt分布（分散は5/3）を、分散1にそろえる
g_t <- sim_garch(z_t, omega = 0.05, alpha = 0.15, beta = 0.80)
fit_n <- fit_garch(g_t$w, dist = "norm")
fit_t <- fit_garch(g_t$w, dist = "std")
data.frame(dist = c("norm", "std"),
           alpha1 = c(fit_n$coef[["alpha1"]], fit_t$coef[["alpha1"]]),
           beta1 = c(fit_n$coef[["beta1"]], fit_t$coef[["beta1"]]),
           shape = c(NA, fit_t$coef[["shape"]]),
           aic = c(fit_n$aic, fit_t$aic),
           kurtosis_z_hat = c(kurtosis(fit_n$e / fit_n$sigma), kurtosis(fit_t$e / fit_t$sigma)))

- `shape`は、推定したt分布の自由度です。本当の値は5です。
- AICは、t分布のほうが小さくなります。
- 標準化残差の尖度は、どちらのモデルでも3よりずっと大きいままです。分布を替えても、残差の裾は軽くなりません。替わるのは、モデルが「裾は重いものだ」と考えるようになる点です。

標準化残差を、正規分布と比べます。点が直線に乗っていれば、正規分布に合っています（Q-Qプロット）。

In [ ]:
ggplot(data.frame(z_hat = fit_n$e / fit_n$sigma), aes(sample = z_hat)) +
  stat_qq(size = 0.8) +
  stat_qq_line(colour = "red") +
  labs(x = "Normal quantile", y = "Standardized residual")

両端で、点が直線から離れます。正規分布が考えるより、極端な値が多く出ています。

### A-8. ARMA-GARCH

ここまでは、系列そのものがホワイトノイズでした。系列に自己相関もあるときは、平均をARMAで、分散をGARCHで表します。これをARMA-GARCHと呼びます。

$$
x_t = \mu + \phi\,(x_{t-1} - \mu) + w_t, \qquad w_t = \sigma_t z_t, \qquad \sigma_t^2 = \omega + \alpha\, w_{t-1}^2 + \beta\, \sigma_{t-1}^2
$$

第3回のAR(1)との違いは、ホワイトノイズ$w_t$がiidではなく、GARCHに従う点だけです。

$\mu = 0.5$、$\phi = 0.3$のAR(1)-GARCH(1,1)を作り、平均を定数だけにしたモデル（`ar = 0`）と、AR(1)にしたモデル（`ar = 1`）を当てはめます。

In [ ]:
set.seed(1)
w <- sim_garch(rnorm(3000), omega = 0.05, alpha = 0.15, beta = 0.80)$w
x <- 0.5 + as.numeric(stats::filter(w, 0.3, method = "recursive"))
fit0 <- fit_garch(x, ar = 0)
fit1 <- fit_garch(x, ar = 1)
round(cbind(true = c(mu = 0.5, ar1 = 0.3, omega = 0.05, alpha1 = 0.15, beta1 = 0.80),
            estimate = fit1$coef, se = fit1$se), 3)
# 標準化残差のリュング・ボックス検定のp値（ラグ10）と、AIC
data.frame(ar = c(0, 1),
           p_ljung_box = c(Box.test(fit0$e / fit0$sigma, lag = 10, type = "Ljung-Box")$p.value,
                           Box.test(fit1$e / fit1$sigma, lag = 10, type = "Ljung-Box", fitdf = 1)$p.value),
           aic = c(fit0$aic, fit1$aic))

平均のモデルが足りないと（`ar = 0`）、標準化残差に自己相関が残ります。平均と分散の両方を、診断で確かめます。

### A-9. 分散の予測を評価する

第2回と同じように、値を隠して、当てて、比べます。最後の部分をテストデータにします。

ただし、本当の分散は観測できません。そこで、次の2つで比べます。

- 残差の2乗$\hat{w}_t^2$との差のRMSE。2乗の期待値が分散なので、2乗を「分散の、ノイズの多い観測値」として使います。
- 95%予測区間$\pm q\,\hat{\sigma}_t$に、実際の値が入った割合（`coverage`）。$q$は$z_t$の分布の97.5%点（正規分布なら1.96）です。予測が合っていれば、0.95に近くなります。

比べる方法は3つです。どれも、テスト期間の毎日について、前の日までの値から1時点先を予測します。係数は、訓練データで推定したものを使い続けます。

- `mean`（平均法）：訓練データの分散。いつも同じ値です。
- `naive`（ナイーブ法）：前の日の残差の2乗。
- `garch`：GARCH(1,1)。

In [ ]:
# z の分布の97.5%点（正規分布なら1.96。t分布は、分散を1にそろえたもの）
q975 <- function(est) {
  if ("shape" %in% names(est)) {
    nu <- est[["shape"]]
    qt(0.975, df = nu) * sqrt((nu - 2) / nu)
  } else {
    qnorm(0.975)
  }
}
# テスト期間（n_train より後）の毎日について、1時点先の分散を3つの方法で予測する
forecast_var <- function(x, n_train, ar = 0, dist = "norm") {
  fit <- fit_garch(x[1:n_train], ar, dist)          # 係数は訓練データだけで推定する
  e <- mean_resid(x, fit$coef)                      # 全期間の残差
  var_train <- mean(e[1:n_train]^2)
  s2 <- garch_filter(e, fit$coef, var0 = var_train)
  test <- (n_train + 1):length(x)
  data.frame(t = test, e = e[test],
             mean  = var_train,                     # 平均法：訓練データの分散
             naive = e[test - 1]^2,                 # ナイーブ法：前の日の残差の2乗
             garch = s2[test],                      # GARCH(1,1)
             q = q975(fit$coef))
}
# 評価：2乗との差のRMSEと、95%予測区間に入った割合
score_var <- function(fc) {
  methods <- c("mean", "naive", "garch")
  data.frame(method = methods,
             RMSE = sapply(methods, function(m) sqrt(mean((fc[[m]] - fc$e^2)^2))),
             coverage = sapply(methods, function(m) mean(abs(fc$e) <= fc$q * sqrt(fc[[m]]))),
             row.names = NULL)
}

A-4の系列（長さ3000）で、最後の1000個をテストデータにします。

In [ ]:
fc <- forecast_var(g$w, n_train = 2000)
score_var(fc)
# 本当の分散が分かっているので、それとの差のRMSEも出せる
sapply(c("mean", "naive", "garch"), function(m) sqrt(mean((fc[[m]] - g$sigma[fc$t]^2)^2)))

- 2乗との差のRMSEは、GARCHが最も小さくなります。ただし、平均法との差は小さく見えます。2乗そのものが、分散のまわりで大きくばらつくからです。
- 本当の分散との差で見ると、GARCHの誤差は、ほかの2つよりずっと小さくなります。
- 区間に入った割合は、平均法とGARCHがどちらも0.95に近くなります。全体の割合だけでは、2つを区別できません。

平均法の区間は、いつも同じ幅です。静かな時期には広すぎ、荒れた時期には狭すぎます。テスト期間を、GARCHの予測した分散が大きい半分と小さい半分に分けて、区間に入った割合を出します。

In [ ]:
fc |>
  mutate(period = ifelse(garch > median(garch), "volatile half", "calm half")) |>
  group_by(period) |>
  summarise(mean = mean(abs(e) <= q * sqrt(mean)),
            garch = mean(abs(e) <= q * sqrt(garch)))

平均法の区間は、静かな時期には入りすぎ、荒れた時期には外れすぎます。GARCHの区間は、どちらの時期でも0.95に近くなります。

## B. 実データを解析する

Rに付属する`EuStockMarkets`を使います。ヨーロッパの4つの株価指数の、日ごとの終値です。

B節は、A節で作った関数（`fit_garch`など）を使います。A節を実行してから進んでください。

### B-0. 設定

このセルの値を変えると、B節の解析がすべて変わります。まずはこのまま実行してください。

In [ ]:
# ここを変えると、以下の解析がすべて変わる
market    <- "DAX"    # 株価指数："DAX"（ドイツ） "SMI"（スイス） "CAC"（フランス） "FTSE"（イギリス）
year_from <- 1991     # 始まりの年：1991〜1993
year_to   <- 1998     # 終わりの年：1996〜1998
ar_order  <- 0        # 平均のモデル：0（定数だけ）か 1（AR(1)）
dist      <- "norm"   # ノイズ z の分布："norm"（正規分布）か "std"（t分布）
h         <- 250      # テストデータの日数：125（約半年）か 250（約1年）

### B-1. データを確かめる

In [ ]:
class(EuStockMarkets)
glimpse(as.data.frame(EuStockMarkets))
tsp(EuStockMarkets)          # 始まりの時点、終わりの時点、1年あたりの観測の数
sum(is.na(EuStockMarkets))

- 1行は1営業日です。列は、DAX（ドイツ）、SMI（スイス）、CAC（フランス）、FTSE（イギリス）の終値です。
- 1860行あります。期間は1991年の半ばから1998年の半ばまでです。
- 土日と祝日は、もともと入っていません。1年を260営業日として、時点に年の目盛りが付けてあります。日付の列はありません。
- 欠測はありません。

### B-2. 前処理

第1回（取引日の通し番号）と同じように、営業日の通し番号を時間の軸にし、対数収益率（%）を作ります。

$$
r_t = 100 \times \left( \log P_t - \log P_{t-1} \right)
$$

そのあと、設定で選んだ指数と期間に絞ります。

In [ ]:
stock <- tibble(time = as.numeric(time(EuStockMarkets)),      # 年の目盛り（1991.5 など）
                price = as.numeric(EuStockMarkets[, market])) |>
  mutate(year = floor(time),
         ret = 100 * difference(log(price))) |>               # 対数収益率（%）
  filter(!is.na(ret), year >= year_from, year <= year_to) |>  # 1行目は前の日がないので外す
  mutate(day = row_number()) |>
  as_tsibble(index = day)
n <- nrow(stock)
n

In [ ]:
ggplot(stock, aes(time, price)) +
  geom_line() +
  labs(x = "Year", y = "Closing price (index points)", title = market)

In [ ]:
ggplot(stock, aes(time, ret)) +
  geom_line() +
  labs(x = "Year", y = "Log return (%)", title = market)

### B-3. 収益率と、その2乗

In [ ]:
stock |>
  ACF(ret, lag_max = 30) |>
  autoplot() +
  labs(x = "Lag h (trading day)", y = "Sample ACF of log return")

In [ ]:
stock |>
  ACF(ret^2, lag_max = 30) |>
  autoplot() +
  labs(x = "Lag h (trading day)", y = "Sample ACF of squared log return")

第1回のgafa_stockと同じ形です。収益率は無相関に近く、2乗には正の自己相関が残ります。

### B-4. 平均のモデルと、ARCH効果の検定

先に、平均のモデル（定数だけ、またはAR(1)）を当てはめ、残差を調べます。

In [ ]:
fit_mean <- arima(stock$ret, order = c(ar_order, 0, 0))
e <- as.numeric(residuals(fit_mean))
# 残差のリュング・ボックス検定、残差の2乗のリュング・ボックス検定、ARCH-LM検定のp値
c(ljung_box    = Box.test(e, lag = 10, type = "Ljung-Box", fitdf = ar_order)$p.value,
  ljung_box_sq = Box.test(e^2, lag = 10, type = "Ljung-Box")$p.value,
  arch_lm      = arch_lm(e)[["p_value"]])

残差の2乗には、はっきりしたARCH効果があります。

### B-5. 金融でない系列と比べる

同じ検定を、金融でない系列に使います。どれもRに付属するデータです。
それぞれにARモデル（次数はAICで選ぶ）を当てはめ、残差にARCH-LM検定を使います。

- `LakeHuron`：ヒューロン湖の水位（年ごと）
- `Nile`：ナイル川の流量（年ごと）
- `lynx`：カナダのオオヤマネコの捕獲数（年ごと、対数）
- `nottem`：ノッティンガムの気温（月ごと）
- `treering`：木の年輪の幅（年ごと、7980年分）

In [ ]:
# ARモデル（次数はAICで選ぶ）を当てはめ、残差を返す
ar_resid <- function(x) {
  e <- as.numeric(ar(x, order.max = 10)$resid)
  e[!is.na(e)]
}

In [ ]:
resids <- list(e, ar_resid(LakeHuron), ar_resid(Nile), ar_resid(log(lynx)),
               ar_resid(nottem), ar_resid(treering))
names(resids) <- c(market, "LakeHuron", "Nile", "log(lynx)", "nottem", "treering")
nonfin <- data.frame(series = names(resids),
                     n = sapply(resids, length),
                     statistic = sapply(resids, function(e) arch_lm(e)[["statistic"]]),
                     p_value = sapply(resids, function(e) arch_lm(e)[["p_value"]]),
                     row.names = NULL)
nonfin$statistic <- round(nonfin$statistic, 1)
nonfin$p_value <- round(nonfin$p_value, 4)
nonfin

`statistic`が検定統計量です。ARCH効果がなければ、自由度5のカイ2乗分布に従い、95%の確率で11.07より小さくなります。

- 水位、流量、捕獲数、気温では、ARCH効果は検出されません。
- 年輪の幅は、p値が小さくなります。

残差の2乗のACFを、3つの系列で並べます。

In [ ]:
pick <- c(market, "nottem", "treering")
sq <- do.call(rbind, lapply(pick, function(s) {
  data.frame(series = s, t = seq_along(resids[[s]]), e2 = resids[[s]]^2)
}))
sq |>
  mutate(series = factor(series, levels = pick)) |>
  as_tsibble(key = series, index = t) |>
  ACF(e2, lag_max = 20) |>
  autoplot() +
  labs(x = "Lag h (step)", y = "Sample ACF of squared residuals")

- 株価指数：多くのラグで、棒が破線の外に出ます。
- 気温：棒は、ほぼ破線の中にあります。
- 年輪の幅：棒が破線の外に出るラグがありますが、棒の高さは株価指数よりずっと低いままです。観測が7980個あり、破線の幅が狭いので、わずかな自己相関でも有意になります。

ボラティリティの集中は、金融データに特有といってよい性質です。金融でない系列では、ARCH効果は出ないか、出ても小さいのがふつうです。
GARCHを使う前に、ARCH効果があるかどうかを、必ず確かめてください。

### B-6. GARCHを当てはめる

平均のモデルとGARCH(1,1)を、同時に推定します。

In [ ]:
fit <- fit_garch(stock$ret, ar = ar_order, dist = dist)
round(cbind(estimate = fit$coef, se = fit$se), 3)
est <- fit$coef
c(persistence = est[["alpha1"]] + est[["beta1"]],
  sd_long = sqrt(est[["omega"]] / (1 - est[["alpha1"]] - est[["beta1"]])),
  sd_sample = sd(stock$ret))

- `persistence`（$\alpha + \beta$）が1に近いほど、荒れた状態が長く続きます。
- `sd_long`は、モデルから計算した、長い目で見た標準偏差です。`sd_sample`（収益率の標本標準偏差）と比べます。

推定したボラティリティ$\hat{\sigma}_t$を、収益率の絶対値に重ねます。

In [ ]:
ggplot(stock |> mutate(sigma = fit$sigma), aes(time)) +
  geom_line(aes(y = abs(ret)), colour = "grey60") +
  geom_line(aes(y = sigma), colour = "red") +
  labs(x = "Year", y = "|log return| (grey) and sigma_t (red), %", title = market)

### B-7. 診断

標準化残差$\hat{z}_t$と、その2乗を調べます。

In [ ]:
z_hat <- fit$e / fit$sigma
plot_acf(z_hat^2, ylab = "Sample ACF of squared standardized residuals")
c(ljung_box    = Box.test(z_hat, lag = 10, type = "Ljung-Box", fitdf = ar_order)$p.value,
  ljung_box_sq = Box.test(z_hat^2, lag = 10, type = "Ljung-Box")$p.value,
  arch_lm      = arch_lm(z_hat)[["p_value"]])

In [ ]:
c(kurtosis_return = kurtosis(stock$ret), kurtosis_z_hat = kurtosis(z_hat))
range(z_hat)                 # 標準化残差の最小と最大
ggplot(data.frame(z_hat = z_hat), aes(sample = z_hat)) +
  stat_qq(size = 0.8) +
  stat_qq_line(colour = "red") +
  labs(x = "Normal quantile", y = "Standardized residual")

平均のモデルとノイズの分布を替えた4つのモデルを、AICで比べます。

In [ ]:
models <- expand.grid(ar = 0:1, dist = c("norm", "std"), stringsAsFactors = FALSE)
models$aic <- sapply(1:4, function(i) fit_garch(stock$ret, models$ar[i], models$dist[i])$aic)
models

### B-8. 分散の予測を評価する

最後の`h`日をテストデータにして、A-9と同じ評価をします。

In [ ]:
fc <- forecast_var(stock$ret, n_train = n - h, ar = ar_order, dist = dist)
acc <- score_var(fc)
acc

In [ ]:
ggplot(fc |> mutate(time = stock$time[t]), aes(time)) +
  geom_line(aes(y = abs(e)), colour = "grey60") +
  geom_line(aes(y = q * sqrt(mean), colour = "mean")) +
  geom_line(aes(y = q * sqrt(garch), colour = "garch")) +
  labs(x = "Year", y = "|residual| (grey) and 95% interval half-width, %", colour = "Method")

灰色の線が色のついた線を上に越えた日が、95%予測区間から外れた日です。

### B-9. 時系列交差検証

訓練データを`h`日ずつ伸ばしながら、同じ評価を3回くり返します。

In [ ]:
fc_cv <- do.call(rbind, lapply(3:1, function(k) {
  x <- stock$ret[1:(n - (k - 1) * h)]              # k = 1 が、B-8と同じ分け方
  cbind(fold = 4 - k, forecast_var(x, n_train = length(x) - h, ar = ar_order, dist = dist))
}))
acc_cv <- score_var(fc_cv)
acc_cv
# 回ごとの、区間に入った割合
fc_cv |>
  group_by(fold) |>
  summarise(mean = mean(abs(e) <= q * sqrt(mean)),
            garch = mean(abs(e) <= q * sqrt(garch)))

読み方：

- B-6：$\hat{\alpha}$は小さく、$\hat{\beta}$は大きくなります。`persistence`は1に近く、荒れた状態も静かな状態も長く続きます。
- B-7：標準化残差の2乗の自己相関は消えます。しかし、尖度は3より大きいままで、Q-Qプロットの両端が直線から離れます。正規分布のGARCHでは、極端な値を説明しきれません。
- B-8、B-9：訓練データが静かな時期で、テストデータが荒れた時期だと、平均法の区間に入る割合は0.95を大きく下回ります。GARCHは平均法より良くなりますが、0.95に届くとは限りません。GARCH(1,1)は、長い目で見た分散を一定と考えているからです。

## C. 設定を変えて回し直す

B-0の設定セルを書き替えて、B節を上からすべて実行し直してください
（Colabでは、設定セルを選んで、メニューの［ランタイム］→［以降のセルを実行］）。

変えられる値：

| 変数 | 値 |
| --- | --- |
| `market` | `"DAX"` `"SMI"` `"CAC"` `"FTSE"` |
| `year_from` | 1991〜1993 |
| `year_to` | 1996〜1998 |
| `ar_order` | 0、1 |
| `dist` | `"norm"` `"std"` |
| `h` | 125、250 |

読むときの観点：

1. 指数を替えると、収益率そのものに自己相関が残る指数はあるか。そのとき、`ar_order`を1にすると、診断とAICはどう変わるか。
2. `dist`を`"std"`にすると、`shape`（t分布の自由度）はいくつになるか。AICと、区間に入る割合はどう変わるか。
3. 終わりの年を1996にすると（荒れた1997〜1998年を外す）、`persistence`と、平均法の区間に入る割合はどう変わるか。
4. `h`を替えると、B-8とB-9で、3つの方法の順位は変わるか。

正解は1つではありません。自分の設定で出た結果を、最初の設定（DAX、1991〜1998年、定数、正規分布、250日）の結果と比べてください。